# 🏆 Churn Prediction: Telco Production Pipeline (V6.4 - FULL RESPONSIBLE AI)
**Dataset**: [blastchar/telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)
**Focus**: SMOTE + High Precision (0.7) + Fairness Audit + SHAP Explainability.

In [ ]:
# 1. Setup & Auth
!pip uninstall umap-learn hdbscan -y -q
!pip install opendatasets mlflow xgboost shap python-dotenv seaborn dagshub scikit-learn==1.5.1 imbalanced-learn -q

import sklearn; print(f"✅ Environment Ready: {sklearn.__version__}")
import os, pandas as pd, numpy as np, mlflow, mlflow.sklearn, matplotlib.pyplot as plt, seaborn as sns, shap, dagshub
import opendatasets as od
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score, precision_recall_curve, recall_score, precision_score, make_scorer
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

mlflow.sklearn.autolog(log_models=False)
dagshub.init(repo_owner="nhannhb92", repo_name="msa24-ddm501-group6-final-project", mlflow=True)

In [ ]:
# 2. Telco Specific Cleaning
def clean_telco_data(df):
    df.columns = [col.lower() for col in df.columns]
    # Remove ID
    df = df.drop(columns=[c for c in ['customerid'] if c in df.columns])
    
    # Totalcharges has spaces for new customers
    df['totalcharges'] = pd.to_numeric(df['totalcharges'], errors='coerce')
    
    # Convert Churn to 1/0
    if 'churn' in df.columns:
        df['churn'] = df['churn'].map({'Yes': 1, 'No': 0})
    
    return df.dropna()

if not os.path.exists("telco-customer-churn"):
    od.download("https://www.kaggle.com/datasets/blastchar/telco-customer-churn")

df_raw = clean_telco_data(pd.read_csv("telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"))
df_train, df_test = train_test_split(df_raw, test_size=0.2, random_state=42, stratify=df_raw['churn'])

print("\n--- 📊 TELCO DATA PROFILE ---")
print(f"Total Records: {len(df_raw)}")
print(f"Churn Rate: {df_raw['churn'].mean():.2%}")
print(f"Features: {list(df_train.columns)}")

In [ ]:
# 3. Full Visual EDA (Telco Edition)
plt.figure(figsize=(18, 5))
plt.subplot(1, 4, 1); sns.countplot(x='churn', data=df_train); plt.title("Class Distribution")
plt.subplot(1, 4, 2); sns.histplot(x='tenure', hue='churn', data=df_train, kde=True); plt.title("Tenure vs Churn")
plt.subplot(1, 4, 3); sns.boxplot(x='churn', y='monthlycharges', data=df_train); plt.title("Monthly Charges")
plt.subplot(1, 4, 4); sns.countplot(x='contract', hue='churn', data=df_train); plt.title("Contract Type")
plt.tight_layout(); plt.show()

In [ ]:
# 4. Custom Scoring Logic (Recall Controlled by Precision Threshold 0.7)
def constrained_recall_scorer(y_true, y_probs, min_precision=0.7):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    idx = np.where(precisions >= min_precision)[0]
    if len(idx) > 0:
        return recalls[idx].max()
    else:
        return 0

recall_ctrl_scorer = make_scorer(constrained_recall_scorer, needs_proba=True, min_precision=0.7)

# 5. Training Engine
def run_telco_experiment(df_tr, df_te, model_type):
    X_tr, y_tr = df_tr.drop(columns=['churn']), df_tr['churn']
    X_te, y_te = df_te.drop(columns=['churn']), df_te['churn']
    
    num_f = X_tr.select_dtypes(include=[np.number]).columns.tolist()
    cat_f = X_tr.select_dtypes(include=['object']).columns.tolist()
    
    pre = ColumnTransformer([
        ('num', Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler())]), num_f),
        ('cat', Pipeline([('imp', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), cat_f)
    ])
    
    # Select Model & Params
    if model_type == "xgboost":
        clf = XGBClassifier(n_estimators=100, max_depth=5, eval_metric='logloss')
        m_p = {'clf__max_depth': [3, 5, 7]}
    elif model_type == "random_forest":
        clf = RandomForestClassifier(random_state=42)
        m_p = {'clf__n_estimators': [100, 200], 'clf__max_depth': [10, 20, None]}
    else:
        clf = LogisticRegression(max_iter=2000)
        m_p = {'clf__C': [0.1, 1.0, 10.0]}
        
    mlflow.set_experiment("Churn_Telco_Production")
    with mlflow.start_run(run_name=f"NB_RAI_{model_type.upper()}"):
        pipe = ImbPipeline([
            ('pre', pre), 
            ('smote', SMOTE(random_state=42)), 
            ('clf', clf)
        ])
        
        search = RandomizedSearchCV(pipe, m_p, n_iter=3, cv=3, scoring=recall_ctrl_scorer, verbose=1)
        search.fit(X_tr, y_tr)
        model = search.best_estimator_
        
        # --- OPTIMAL THRESHOLD (Precision 0.7 floor) ---
        y_prb = model.predict_proba(X_te)[:, 1]
        precisions, recalls, thresholds = precision_recall_curve(y_te, y_prb)
        min_precision = 0.7
        idx = np.where(precisions >= min_precision)[0]
        
        if len(idx) > 0 and idx[0] < len(thresholds):
            best_idx = idx[np.argmax(recalls[idx])]
            best_threshold = thresholds[min(best_idx, len(thresholds)-1)]
        else:
            best_threshold = 0.5
            
        y_prd_custom = (y_prb >= best_threshold).astype(int)
        
        print(f"\n🏆 TELCO REPORT ({model_type.upper()}):")
        print(f"Optimal Training Score: {search.best_score_:.4f}")
        print(f"Applied Threshold: {best_threshold:.4f}")
        print(classification_report(y_te, y_prd_custom))
        
        mlflow.log_metric("final_recall", recall_score(y_te, y_prd_custom))
        mlflow.log_metric("final_precision", precision_score(y_te, y_prd_custom))
        mlflow.log_param("best_threshold", best_threshold)
        
        # --- RESPONSIBLE AI: FAIRNESS AUDIT ---
        res = df_te.copy(); res['pred'] = y_prd_custom
        # Avoid errors if gender column is missing or lowercase issues
        g_col = 'gender' if 'gender' in res.columns else ('Gender' if 'Gender' in res.columns else None)
        if g_col:
            audit = res.groupby(g_col).agg(Actual=('churn', 'mean'), Predicted=('pred', 'mean'))
            print(f"⚖️ Fairness Audit ({g_col}):\n{audit}")
            
            if len(audit) >= 2:
                bias_gap = abs(audit.iloc[0, 1] - audit.iloc[1, 1])
                mlflow.log_metric("bias_gap_gender", bias_gap)
                print(f"   - Logged Bias Gap: {bias_gap:.4f}")
            
            audit.to_csv(f"audit_{model_type}.csv")
            mlflow.log_artifact(f"audit_{model_type}.csv")
        
        # --- RESPONSIBLE AI: EXPLAINABILITY (SHAP) ---
        try:
            print(f"🔍 SHAP Analysis for {model_type}...")
            X_t = model.named_steps['pre'].transform(X_te)
            if hasattr(X_t, "toarray"): X_t = X_t.toarray()
            
            # Tree-based vs Linear explainer handling
            if model_type in ["xgboost", "random_forest"]:
                explainer = shap.Explainer(model.named_steps['clf'])
            else:
                explainer = shap.Explainer(model.named_steps['clf'], X_t)
                
            shap_v = explainer(X_t)
            
            plt.figure(figsize=(10, 6))
            # Handle SHAP multi-output for RF if necessary
            shap_plot_data = shap_v.values if not isinstance(shap_v.values, list) else shap_v.values[1]
            
            shap.summary_plot(shap_v, X_t, feature_names=model.named_steps['pre'].get_feature_names_out(), show=False)
            plt.tight_layout()
            plt.savefig(f"shap_{model_type}.png")
            mlflow.log_artifact(f"shap_{model_type}.png")
            plt.show()
        except Exception as e: 
            print(f"⚠️ SHAP failed for {model_type}: {e}")

        # Register
        mlflow.sklearn.log_model(model, "model", registered_model_name=f"Churn_{model_type}_RAI")
        print(f"✅ {model_type.upper()} Registered with RAI Artifacts")

for m in ["xgboost", "random_forest", "logistic_regression"]:
    run_telco_experiment(df_train, df_test, m)